# LLaMA：因果语言模型（含 GQA）

这个 Notebook 从零实现 LLaMA 2/3 风格的因果语言模型，并在 TinyShakespeare 上训练字符级 LM。

内容包括：
- RMSNorm（无均值归一化，比 LayerNorm 更简洁）
- RoPE（旋转位置编码，相对位置感知）
- **GQA（Grouped Query Attention）**：n_kv_heads < n_heads，KV Cache 更小
- SwiGLU 激活（门控线性单元变体）
- Pre-Norm Decoder-only Transformer
- 因果语言模型训练
- 温度采样 / Top-k 文本生成

## 1. 环境准备

只需 PyTorch，无其他依赖：

```bash
pip install torch
```

In [ ]:
import math
import requests
from dataclasses import dataclass, field
from typing import Tuple

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

plt.style.use('seaborn-v0_8')
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
@dataclass
class Config:
    # 数据
    data_url: str  = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
    seq_len: int   = 256
    batch_size: int = 64
    # 模型（LLaMA-tiny，适合单卡或 CPU 演示）
    vocab_size: int  = 65    # 字符级词表，TinyShakespeare 约 65 个字符
    n_layers: int    = 6
    d_model: int     = 256
    n_heads: int     = 8
    n_kv_heads: int  = 2    # GQA：2 个 KV 头，每组 4 个 Q 头共享一对 KV
    d_ff: int        = 512
    dropout: float   = 0.1
    max_seq_len: int = 2048  # RoPE 预计算上限
    # 训练
    lr: float    = 3e-4
    epochs: int  = 5


cfg = Config()
cfg

## 2. 数据集：TinyShakespeare（字符级）

In [ ]:
# 下载 TinyShakespeare（约 1MB 纯文本）
try:
    text = requests.get(cfg.data_url, timeout=10).text
    print(f'Downloaded: {len(text):,} characters')
except Exception:
    # 网络不可用时使用内联小样本
    text = 'To be or not to be, that is the question. ' * 2000
    print('Using fallback text.')

# 字符级词表
chars   = sorted(set(text))
char2id = {c: i for i, c in enumerate(chars)}
id2char = {i: c for c, i in char2id.items()}
vocab_size = len(chars)

# 更新 Config 的实际词表大小
cfg.vocab_size = vocab_size

print(f'Vocab size: {vocab_size}')
print(f'Sample: {repr(text[:80])}')

# 编解码函数
encode = lambda s: [char2id[c] for c in s]
decode = lambda ids: ''.join(id2char[i] for i in ids)

# 整体 encode
data = torch.tensor(encode(text), dtype=torch.long)
n    = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

In [ ]:
class CharDataset(Dataset):
    def __init__(self, data, seq_len):
        self.data    = data
        self.seq_len = seq_len

    def __len__(self):
        return len(self.data) - self.seq_len

    def __getitem__(self, idx):
        chunk = self.data[idx: idx + self.seq_len + 1]
        return chunk[:-1], chunk[1:]  # (input, target)


train_ds     = CharDataset(train_data, cfg.seq_len)
val_ds       = CharDataset(val_data,   cfg.seq_len)
train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=cfg.batch_size, shuffle=False)

x, y = next(iter(train_loader))
print('input shape:', x.shape, '  target shape:', y.shape)

## 3. LLaMA 架构实现

In [ ]:
class RMSNorm(nn.Module):
    """只做 RMS 缩放，不减均值，比 LayerNorm 更简洁，效果相当。"""
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps    = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        rms = x.float().pow(2).mean(-1, keepdim=True).add(self.eps).sqrt()
        return (x.float() / rms * self.weight).to(x.dtype)

In [ ]:
def precompute_freqs_cis(dim, max_seq_len, theta=10000.0):
    # RoPE 频率：偶数维用不同频率的正弦余弦对
    freqs  = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
    t      = torch.arange(max_seq_len)
    freqs  = torch.outer(t, freqs)          # (max_seq_len, dim/2)
    # 用复数表示旋转，cos+i*sin 更紧凑
    return torch.polar(torch.ones_like(freqs), freqs)  # complex64


def apply_rotary_emb(q, k, freqs_cis):
    # q/k: (B, heads, seq_len, head_dim) → reshape 成复数再旋转
    def rotate(x):
        x_c = torch.view_as_complex(x.float().reshape(*x.shape[:-1], -1, 2))
        return torch.view_as_real(x_c * freqs_cis).flatten(-2).to(x.dtype)
    return rotate(q), rotate(k)

In [ ]:
def repeat_kv(kv, n_rep):
    """把 KV 头重复 n_rep 次对齐 Q 头数，实现 GQA 的核心操作。"""
    if n_rep == 1:
        return kv
    B, n_kv, seq, head_dim = kv.shape
    return kv.unsqueeze(2).expand(B, n_kv, n_rep, seq, head_dim).reshape(B, n_kv * n_rep, seq, head_dim)


class GroupedQueryAttention(nn.Module):
    """
    Grouped Query Attention（GQA）：
    - n_heads Q 头，n_kv_heads K/V 头（n_kv_heads 是 n_heads 的因数）
    - 每组 n_heads // n_kv_heads 个 Q 共享一对 K/V
    - KV Cache 大小 = n_kv_heads，显著小于 MHA 的 n_heads

    参数对比（d_model=256, n_heads=8）：
      MHA  (n_kv_heads=8): Q+K+V 权重 = 3 × 256 × 256 = 196,608
      GQA  (n_kv_heads=2): Q+K+V 权重 = (8+2+2)/8 × 196,608 ≈ 147,456
      MQA  (n_kv_heads=1): Q+K+V 权重 = (8+1+1)/8 × 196,608 ≈ 98,304
    """
    def __init__(self, cfg):
        super().__init__()
        assert cfg.n_heads % cfg.n_kv_heads == 0
        self.n_heads    = cfg.n_heads
        self.n_kv_heads = cfg.n_kv_heads
        self.n_rep      = cfg.n_heads // cfg.n_kv_heads  # 每组 Q 头数
        self.head_dim   = cfg.d_model // cfg.n_heads

        self.wq = nn.Linear(cfg.d_model, cfg.n_heads    * self.head_dim, bias=False)
        self.wk = nn.Linear(cfg.d_model, cfg.n_kv_heads * self.head_dim, bias=False)
        self.wv = nn.Linear(cfg.d_model, cfg.n_kv_heads * self.head_dim, bias=False)
        self.wo = nn.Linear(cfg.d_model, cfg.d_model,                    bias=False)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x, freqs_cis, mask):
        B, T, _ = x.shape
        q = self.wq(x).view(B, T, self.n_heads,    self.head_dim).transpose(1, 2)
        k = self.wk(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.wv(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)

        # 对 Q、K 施加旋转位置编码
        q, k = apply_rotary_emb(q, k, freqs_cis[:T])

        # 把 KV 重复以对齐 Q 头数（GQA 核心）
        k = repeat_kv(k, self.n_rep)
        v = repeat_kv(v, self.n_rep)

        # 缩放点积注意力（加因果 mask）
        scale = self.head_dim ** -0.5
        attn  = (q * scale) @ k.transpose(-2, -1) + mask[:T, :T]
        attn  = F.softmax(attn.float(), dim=-1).to(q.dtype)
        attn  = self.drop(attn)

        out = (attn @ v).transpose(1, 2).reshape(B, T, -1)
        return self.wo(out)

In [ ]:
class SwiGLU(nn.Module):
    """LLaMA 的 FFN：SiLU(W1·x) ⊙ (W3·x)，再接 W2。比 GeLU FFN 效果更好。"""
    def __init__(self, cfg):
        super().__init__()
        self.w1 = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.w2 = nn.Linear(cfg.d_ff,   cfg.d_model, bias=False)
        self.w3 = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))


class LlamaBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = RMSNorm(cfg.d_model)
        self.attn  = GroupedQueryAttention(cfg)
        self.norm2 = RMSNorm(cfg.d_model)
        self.ffn   = SwiGLU(cfg)

    def forward(self, x, freqs_cis, mask):
        # Pre-Norm + 残差
        x = x + self.attn(self.norm1(x), freqs_cis, mask)
        x = x + self.ffn(self.norm2(x))
        return x

In [ ]:
class LlamaModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg    = cfg
        self.embed  = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.drop   = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([LlamaBlock(cfg) for _ in range(cfg.n_layers)])
        self.norm   = RMSNorm(cfg.d_model)
        self.head   = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        # 权重共享：embedding 和 lm head 使用同一矩阵
        self.head.weight = self.embed.weight

        # RoPE 频率预计算（注册为 buffer，不参与梯度）
        freqs = precompute_freqs_cis(cfg.d_model // cfg.n_heads, cfg.max_seq_len)
        self.register_buffer('freqs_cis', freqs)

        # 因果掩码（上三角 -inf）
        causal = torch.full((cfg.max_seq_len, cfg.max_seq_len), float('-inf'))
        causal = torch.triu(causal, diagonal=1)
        self.register_buffer('causal_mask', causal)

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, idx):
        B, T = idx.shape
        x    = self.drop(self.embed(idx))
        for block in self.blocks:
            x = block(x, self.freqs_cis, self.causal_mask)
        x = self.norm(x)
        return self.head(x)  # (B, T, vocab_size)


model = LlamaModel(cfg).to(device)
model

## 4. GQA vs MHA vs MQA 对比

In [ ]:
def kv_cache_size(n_kv_heads, head_dim, seq_len, n_layers, dtype_bytes=2):
    # KV Cache = 2(K+V) × n_kv_heads × head_dim × seq_len × n_layers × bytes
    return 2 * n_kv_heads * head_dim * seq_len * n_layers * dtype_bytes


head_dim = cfg.d_model // cfg.n_heads
seq_len  = 2048

print(f'KV Cache 大小对比（seq_len={seq_len}, n_layers={cfg.n_layers}, dtype=fp16）：\n')
for variant, n_kv in [('MHA (n_kv=8)', cfg.n_heads), ('GQA (n_kv=2)', 2), ('MQA (n_kv=1)', 1)]:
    size_mb = kv_cache_size(n_kv, head_dim, seq_len, cfg.n_layers) / 1024**2
    print(f'  {variant:20s}: {size_mb:.2f} MB')

## 5. Token 序列尺寸分析

In [ ]:
@torch.no_grad()
def inspect_shapes(model, cfg):
    idx = torch.randint(0, cfg.vocab_size, (2, cfg.seq_len))
    print(f'input (idx)    : {tuple(idx.shape)}')
    x = model.embed(idx)
    print(f'embedding      : {tuple(x.shape)}')
    # 追踪单个 block 内部
    blk = model.blocks[0]
    q   = blk.attn.wq(blk.norm1(x))
    k   = blk.attn.wk(blk.norm1(x))
    B, T = x.shape[:2]
    q_h = q.view(B, T, cfg.n_heads,    -1).transpose(1, 2)
    k_h = k.view(B, T, cfg.n_kv_heads, -1).transpose(1, 2)
    print(f'Q heads        : {tuple(q_h.shape)}  (B, n_heads,    T, head_dim)')
    print(f'K heads (GQA)  : {tuple(k_h.shape)}  (B, n_kv_heads, T, head_dim)')
    k_rep = repeat_kv(k_h, cfg.n_heads // cfg.n_kv_heads)
    print(f'K after repeat : {tuple(k_rep.shape)}  (B, n_heads,    T, head_dim)')
    out = model(idx)
    print(f'logits         : {tuple(out.shape)}')


inspect_shapes(model.cpu(), cfg)
model = model.to(device)

## 6. 关键机制解读

### RMSNorm vs LayerNorm
- LayerNorm：减均值 + 除方差，两步操作。
- RMSNorm：只除 RMS（均方根），无减均值，计算更快，经验上效果相当。

### RoPE vs 绝对位置编码
- GPT-2 学可学习绝对位置 embedding，外推能力弱。
- RoPE 把位置编码进 Q/K 的旋转变换，天然捕获相对位置，外推能力更强。

### GQA 的动机
- 推理时 KV Cache 是主要显存瓶颈：`2 × n_heads × head_dim × seq_len × layers × bytes`。
- GQA 把 KV 头数从 n_heads 减到 n_kv_heads，Cache 缩小 n_heads/n_kv_heads 倍，质量损失很小。

### SwiGLU vs GeLU FFN
- GPT-2：`FFN(x) = GeLU(W1·x)·W2`（两个权重矩阵）
- LLaMA：`FFN(x) = SiLU(W1·x) ⊙ (W3·x)·W2`（三个权重矩阵，门控结构）
- 门控让模型能自适应选择激活哪些特征，实验上优于 GeLU。

### Pre-Norm 架构
- 先归一化再做 Attention/FFN（Post-Norm 是先做再归一化）。
- Pre-Norm 训练更稳定，梯度不容易消失或爆炸。

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


total = count_parameters(model)
attn  = sum(p.numel() for n, p in model.named_parameters() if 'attn' in n)
ffn   = sum(p.numel() for n, p in model.named_parameters() if 'ffn'  in n)
print(f'Total       : {total:,}')
print(f'Attention   : {attn:,}  ({100*attn/total:.1f}%)')
print(f'FFN (SwiGLU): {ffn:,}  ({100*ffn/total:.1f}%)')

## 7. 训练函数

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=0.1)


def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0
    total      = 0
    for x, y in loader:
        x, y   = x.to(device), y.to(device)
        logits = model(x)
        loss   = criterion(logits.view(-1, logits.size(-1)), y.view(-1))
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        total      += x.size(0)
    return total_loss / total


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total_loss = 0.0
    total      = 0
    for x, y in loader:
        x, y   = x.to(device), y.to(device)
        logits = model(x)
        loss   = criterion(logits.view(-1, logits.size(-1)), y.view(-1))
        total_loss += loss.item() * x.size(0)
        total      += x.size(0)
    return total_loss / total

## 8. 训练主循环

In [ ]:
history = {'train_loss': [], 'val_loss': []}

for epoch in range(cfg.epochs):
    train_loss = train_one_epoch(model, train_loader, optimizer, device)
    val_loss   = evaluate(model, val_loader, device)
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    print(
        f'Epoch [{epoch + 1}/{cfg.epochs}]  '
        f'train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  '
        f'train_ppl={math.exp(train_loss):.2f}  val_ppl={math.exp(val_loss):.2f}'
    )

In [ ]:
epochs_r = range(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_r, history['train_loss'], label='train')
axes[0].plot(epochs_r, history['val_loss'],   label='val')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(epochs_r, [math.exp(l) for l in history['train_loss']], label='train')
axes[1].plot(epochs_r, [math.exp(l) for l in history['val_loss']],   label='val')
axes[1].set_title('Perplexity')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.show()

## 9. 文本生成

In [ ]:
@torch.no_grad()
def generate(model, prompt, max_new_tokens=200, temperature=1.0, top_k=50, device='cpu'):
    model.eval()
    idx = torch.tensor(encode(prompt), dtype=torch.long, device=device).unsqueeze(0)

    for _ in range(max_new_tokens):
        # 超过最大长度时截断
        idx_cond = idx[:, -cfg.seq_len:]
        logits   = model(idx_cond)[:, -1, :]  # 只取最后一步的 logit

        # 温度控制采样分布的平坦程度
        logits = logits / temperature

        # Top-k 过滤
        if top_k > 0:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, -1:]] = float('-inf')

        probs     = F.softmax(logits, dim=-1)
        next_tok  = torch.multinomial(probs, num_samples=1)
        idx       = torch.cat([idx, next_tok], dim=1)

    return decode(idx[0].tolist())


prompts = ['HAMLET:', 'First Citizen:', 'To be']

print('=== temperature=0.8, top_k=40 ===\n')
for p in prompts:
    out = generate(model, p, max_new_tokens=150, temperature=0.8, top_k=40, device=device)
    print(out)
    print('-' * 60)